# Pilot 4 · Behavioural follow-up

Saved answers → verified source bundle → harmonized scoring report.

**CPU only.** This release implements source/scoring, paired transitions and sampled reliability (Tickets 01–03). Human audit arrives later. No training or new generation runs here.

**Local release:** the pinned source has not been published to GitHub. Run locally, or supply the exact checkout in Colab; this is not a ready public Colab link.

**Resume:** run 1–2, then 4 to continue scoring, 5 for score tables, 6 for transitions or 7 for sampled reliability. Source selection is needed only when importing a new bundle.

See `docs/pilot4-behaviour-breakdown.md` for the manifest contract and source-evidence limits.


## 1. Load the CPU harness

Use the repository checkout and existing Python/NumPy environment. Colab also mounts Drive. No Torch, Transformers or CUDA setup is needed.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

HARNESS_COMMIT = '62d5608f9598412936af554ae8e44c8e38495876'
if IN_COLAB:
    # Supply a checkout of this local release; it is not published to GitHub yet.
    REPO_DIR = Path(os.environ.get('SPAR_REPO_DIR', '/content/pilot4-followup-harness'))
    if not (REPO_DIR / 'pilot_eval').is_dir():
        raise RuntimeError('Upload the local release and set SPAR_REPO_DIR, or publish the pinned release before using Colab.')
    actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    if actual != HARNESS_COMMIT:
        raise RuntimeError('Checkout differs from the notebook source pin; select the pinned release.')
else:
    REPO_DIR = Path(os.environ.get('SPAR_REPO_DIR', Path.cwd())).resolve()
    if not (REPO_DIR / 'pilot_eval').is_dir():
        raise RuntimeError('Start Jupyter in the repository root or set SPAR_REPO_DIR.')

loaded = sys.modules.get('pilot_eval')
if loaded and not Path(loaded.__file__).resolve().is_relative_to(REPO_DIR):
    raise RuntimeError('Restart the kernel: another harness checkout is already imported.')
sys.path.insert(0, str(REPO_DIR))
from pilot_eval.notebook_session import NotebookSession

print('CPU saved-output workflow. No model libraries or GPU are required.')


## 2. Select the analysis and restore its saved paths

A new name creates a separate immutable analysis. Reuse the same name to resume verified work.


In [ ]:
ANALYSIS_NAME = 'pilot4-behaviour-breakdown-v1'  # @param {type:"string"}
SOURCE_SELECTION_PATH = ''  # @param {type:"string"}

ANALYSIS_NAME = os.environ.get('SPAR_BEHAVIOUR_NAME', ANALYSIS_NAME)
default_root = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if IN_COLAB else REPO_DIR / 'artifacts'
ARTIFACT_ROOT = Path(os.environ.get('SPAR_BEHAVIOUR_ROOT', default_root)).resolve()
runner = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=ANALYSIS_NAME)
saved = NotebookSession.saved_selection(ARTIFACT_ROOT, ANALYSIS_NAME)
SOURCE_SELECTION_PATH = os.environ.get('SPAR_BEHAVIOUR_SELECTION', SOURCE_SELECTION_PATH or saved.get('behaviour_selection', ''))
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
print('Saved plan:', CONFIG_PATH, '| exists:', CONFIG_PATH.is_file())


## 3. Verify and import explicit source exports

Reads the selected files, checks identities and recognized stored scores, and copies a source-bound bundle. Existing evidence remains unchanged. Missing provenance is reported rather than inferred.


In [ ]:
if SOURCE_SELECTION_PATH and Path(SOURCE_SELECTION_PATH).is_file():
    runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-prepare',
                '--selection', SOURCE_SELECTION_PATH, '--name', ANALYSIS_NAME,
                '--output-root', str(ARTIFACT_ROOT)])
    runner.save_selection({'behaviour_selection': SOURCE_SELECTION_PATH})
elif CONFIG_PATH.is_file():
    print('Restored existing plan; scoring/verification will check its saved inputs.')
else:
    raise ValueError('Set SOURCE_SELECTION_PATH to the explicit source manifest described in the workflow guide.')


## 4. Score or resume endpoints

Produces strict, flexible-v3 and last-number decisions. Completed sealed units are reused; altered inputs or outputs stop the command.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-score',
            '--config', str(CONFIG_PATH), '--output-root', str(ARTIFACT_ROOT)])


## 5. Verify and read the scoring report

Shows original denominators, extractability, historical-score evidence and source compatibility. This is not yet the transition or human-audit report.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-verify',
            '--config', str(CONFIG_PATH), '--output-root', str(ARTIFACT_ROOT)])
config = json.loads(CONFIG_PATH.read_text())
run_directory = runner.artifact(config['run_path'])
report_text = (run_directory / 'results/report.md').read_text()
print(report_text)
print('Structured results:', run_directory / 'results/results.json')
print('Raw and harmonized records:', run_directory / 'results/responses.jsonl')


## 6. Paired greedy transitions (CPU)

Uses the verified scoring bundle. Saves correct/wrong/invalid flows, gains and losses, paired intervals, direct GRPO64–SFT64 comparison and SFT length descriptions. Re-running verifies and restores completed outputs.


In [ ]:
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
TRANSITION_NAME = ANALYSIS_NAME + '-transitions-v1'
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-transitions',
            '--config', str(CONFIG_PATH), '--name', TRANSITION_NAME,
            '--output-root', str(ARTIFACT_ROOT)])
transition_directory = runner.artifact('reports/' + TRANSITION_NAME)
print((transition_directory / 'results/report.md').read_text())
print('Per-question evidence:', transition_directory / 'results/paired-items.jsonl')
print('Plot:', transition_directory / 'plots/accuracy-differences.svg')


## 7. Observed eight-draw GRPO reliability (CPU)

Choose the frozen native GRPO source index from the inventory below. The source must include all eight saved held-out draws at steps 0 and 64. This measures observed coverage and mean reward; it does not generate responses or establish new ability. The choice is saved for the next session.


In [ ]:
SAMPLED_SOURCE_INDEX = -1  # @param {type:"integer"}
CONFIG_PATH = runner.plan(ANALYSIS_NAME, 'behaviour.analysis.json')
source_plan = json.loads(CONFIG_PATH.read_text())
for index, source in enumerate(source_plan['sources']):
    print(index, '|', source['method'], '|', source['format'])
saved_sampling = NotebookSession.saved_selection(ARTIFACT_ROOT, ANALYSIS_NAME)
selected_index = int(os.environ.get('SPAR_SAMPLED_SOURCE_INDEX',
    saved_sampling.get('sampled_source_index', SAMPLED_SOURCE_INDEX) if SAMPLED_SOURCE_INDEX < 0 else SAMPLED_SOURCE_INDEX))
if selected_index < 0:
    raise ValueError('Set SAMPLED_SOURCE_INDEX to the native GRPO source shown above.')
SAMPLING_NAME = ANALYSIS_NAME + '-sampling-v1'
runner.run([sys.executable, '-u', '-m', 'pilot_eval', 'behaviour-sampling',
            '--config', str(CONFIG_PATH), '--sampled-source-index', str(selected_index),
            '--name', SAMPLING_NAME, '--output-root', str(ARTIFACT_ROOT)])
runner.save_selection({'sampled_source_index': selected_index})
sampling_directory = runner.artifact('reports/' + SAMPLING_NAME)
print((sampling_directory / 'results/report.md').read_text())
print('Per-question counts:', sampling_directory / 'results/per-item.jsonl')
print('Plot:', sampling_directory / 'plots/observed-coverage.svg')
